# 1. Project Title: Plant Disease Detection (CNN and Transfer Learning)

**Kit:** AI Practical Project Learning Kit · **Project 2 of 5** · **Domain:** Computer Vision and Deep Learning · **Difficulty:** Beginner to Intermediate

**Use a GPU:** `Runtime → Change runtime type → T4 GPU` (free). Training on CPU works but is many times slower.

> **Educational use only.** This is a classroom image classifier trained on lab-style photos. Its predictions are **not a substitute for expert agricultural diagnosis**. If a real crop is sick, consult an agronomist or your local agriculture extension service.

## 2. Project Overview

You will teach a computer to look at a photo of a single plant leaf and say **which plant it is and whether it is healthy or which disease it shows**. You will build a small **CNN from scratch** to learn how convolutional networks work, and then use **transfer learning with MobileNetV2**, a network pre-trained by Google on 1.4 million everyday photos, and see how much a borrowed brain helps.

## 3. Real-World Problem Statement

Plant diseases cost farmers a large part of their harvest each year, and early signs are visible on leaves. Expert inspection is not always available to small farms. An image classifier can act as a **first-pass screening aid** that tells a farmer "this looks like early blight, please get it checked".

**Our task:** given a leaf photo, predict one of the 38 classes in the PlantVillage dataset (for example `Tomato — Late blight`, `Apple — healthy`).

## 4. Project Objectives

1. Explain image classification and how a computer sees an image (pixels and channels).
2. Load, resize and normalize images; build training / validation / test sets.
3. Explain convolution, pooling, flattening, dense layers and activation functions.
4. Build a CNN from scratch and train it.
5. Use **transfer learning** with MobileNetV2 and compare the two models.
6. Interpret accuracy/loss curves, a confusion matrix and a classification report.
7. Classify a leaf image that you upload and read the confidence honestly.

## 5. What We Will Build

`Leaf photo → resize to 128×128 → CNN / MobileNetV2 → 38 class probabilities → predicted plant + condition + confidence`

## 6. Technologies and Libraries Used

| Library | Purpose |
|---|---|
| `tensorflow` / `keras` | Neural networks, MobileNetV2 and training |
| `tensorflow-datasets` | Downloads the PlantVillage dataset |
| `numpy` | Image arrays |
| `Pillow` | Opening and resizing your own image |
| `matplotlib` | Charts and image grids |
| `scikit-learn` | Data split, confusion matrix, classification report |

## 7. System Architecture

```
PlantVillage (54,303 leaf images, 38 classes)
      │  keep up to N images per class, resize to 128x128
      ▼
 Split: 70% train / 15% validation / 15% test (stratified)
      │
      ├──► Model A: CNN from scratch  (Conv-Pool x3 → Flatten → Dense)
      └──► Model B: MobileNetV2 (frozen, ImageNet) + new Dense head
                          │  data augmentation during training
                          ▼
        Evaluate on TEST: accuracy · confusion matrix · report
                          ▼
        Upload leaf → predict class + confidence (+ warnings)
```

## 8. Understanding the Dataset

**PlantVillage** is an open dataset from Penn State University / EPFL of **54,303 photos of single leaves** on a plain background, in **38 classes** that combine a crop and a condition (healthy or a disease) across 14 crops (apple, corn, grape, potato, tomato and others).

Two honest facts to remember:

* The photos were taken in **controlled conditions** (one leaf, plain background). Models trained on it often drop in accuracy on photos taken in a real field.
* Some classes have thousands of images, others only a few hundred, so the dataset is **imbalanced**. To keep this notebook fast and fair we keep at most `MAX_PER_CLASS` images per class.

## 9. Dataset Download / Loading

The first run downloads the dataset through TensorFlow Datasets (about 0.8 GB), which can take several minutes in Colab. It is cached afterwards. We then stream through it **once**, keeping a balanced subset resized to 128×128.

In [ ]:
# Section 9a - install (if needed) and imports
!pip install -q tensorflow-datasets

import json, collections, time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

SEED = 42
keras.utils.set_random_seed(SEED)
IMG = 128                 # image side in pixels (MobileNetV2 accepts 96, 128, 160, 192, 224)
MAX_PER_CLASS = 200       # images kept per class (raise it for better accuracy; lower it for speed)
print("TensorFlow", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

In [ ]:
# Section 9b - download and stream the dataset once
ds, info = tfds.load("plant_village", split="train", with_info=True)
class_names = info.features["label"].names
NUM_CLASSES = len(class_names)
print("Total images:", info.splits["train"].num_examples, "| classes:", NUM_CLASSES)

kept = collections.Counter()      # images kept per class
total = collections.Counter()     # all images per class (to show the imbalance)
imgs, labels = [], []
t0 = time.time()
for ex in ds.prefetch(tf.data.AUTOTUNE):
    lab = int(ex["label"].numpy())
    total[lab] += 1
    if kept[lab] < MAX_PER_CLASS:
        small = tf.image.resize(ex["image"], (IMG, IMG), antialias=True)
        imgs.append(tf.cast(tf.round(small), tf.uint8).numpy())
        labels.append(lab); kept[lab] += 1
X = np.stack(imgs); y = np.array(labels)
print(f"Kept {len(X)} images in {time.time() - t0:.0f}s | array shape {X.shape} | dtype {X.dtype}")

## 10. Dataset Exploration

In [ ]:
# Section 10a - class names and the imbalance in the full dataset
def pretty(name):                   # 'Tomato___Late_blight' -> 'Tomato - Late blight'
    return name.replace("___", " - ").replace("_", " ")
pretty_names = [pretty(n) for n in class_names]

order = np.argsort([total[i] for i in range(NUM_CLASSES)])
plt.figure(figsize=(8, 10))
plt.barh([pretty_names[i] for i in order], [total[i] for i in order], color="seagreen")
plt.title("Images per class in the FULL dataset"); plt.xlabel("number of images"); plt.tight_layout(); plt.show()
print("Smallest class:", min(total.values()), "images | largest class:", max(total.values()), "images")
print("Classes with fewer than", MAX_PER_CLASS, "images:", sum(1 for i in range(NUM_CLASSES) if total[i] < MAX_PER_CLASS))

The huge difference between classes is why we cap images per class: otherwise a model could score well just by favouring the biggest classes.

In [ ]:
# Section 10b - look at some leaves
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for ax, i in zip(axes.ravel(), rng.choice(len(X), 18, replace=False)):
    ax.imshow(X[i]); ax.set_title(pretty_names[y[i]], fontsize=8); ax.axis("off")
plt.suptitle("Random samples (resized to %dx%d)" % (IMG, IMG)); plt.tight_layout(); plt.show()

print("One image is an array of shape", X[0].shape, "= height x width x 3 colour channels (R, G, B), values 0-255.")

## 11. Data Preprocessing

* **Resizing:** networks need every image to be the same size; we already resized to 128×128 (which also makes training faster).
* **Normalization:** pixel values 0–255 are rescaled inside the model (`Rescaling` layer): to 0–1 for the CNN and to −1…+1 for MobileNetV2, which is what it was trained with.
* **Stratified split:** 70 % train, 15 % validation, 15 % test, keeping the same class proportions in each. The **test set stays unseen until the end**.
* **Data augmentation:** random flips, rotations and zooms create new variations of training images, so the model cannot just memorise them. It is active **only during training**.

In [ ]:
# Section 11 - split and build tf.data pipelines
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp)
print("Train:", len(X_tr), "| Validation:", len(X_val), "| Test:", len(X_te))

BATCH = 32
def make_ds(Xa, ya, training):
    d = tf.data.Dataset.from_tensor_slices((Xa, ya))
    if training:
        d = d.shuffle(len(Xa), seed=SEED)
    d = d.batch(BATCH).map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=tf.data.AUTOTUNE)
    return d.prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(X_tr, y_tr, True), make_ds(X_val, y_val, False), make_ds(X_te, y_te, False)

augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
], name="augmentation")

# Show what augmentation does to one image
sample = tf.cast(X_tr[:1], tf.float32)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(X_tr[0]); axes[0].set_title("original"); axes[0].axis("off")
for ax in axes[1:]:
    ax.imshow(augment(sample, training=True)[0].numpy().astype("uint8")); ax.set_title("augmented"); ax.axis("off")
plt.show()

## 12. Understanding the Algorithm

### How a CNN sees a leaf
* **Convolution layer:** a small window (a *filter*, for example 3×3) slides across the image. At each position it multiplies the pixels by its weights and adds them up. Early filters learn simple things (edges, colour blobs); deeper filters combine them into spots, veins and lesions. `Conv2D(32, 3)` means 32 different filters.
* **Activation (ReLU):** `max(0, x)` after each convolution adds the non-linearity needed to learn complex patterns. The final layer uses **softmax**, which turns scores into probabilities that add up to 1.
* **Pooling (`MaxPooling2D`):** keeps the strongest value in each 2×2 block, halving the image. This cuts computation and makes the network less sensitive to exactly where a spot is.
* **Flatten:** unrolls the final feature maps into one long vector.
* **Dense layers:** ordinary fully-connected neurons that combine all the features to decide the class.
* **Dropout:** randomly switches off some neurons during training to reduce overfitting.

### Transfer learning
Training a large CNN needs lots of data and compute. **MobileNetV2** has already learned general visual features (edges, textures, shapes) from the **ImageNet** dataset. We **freeze** it (its weights do not change), remove its original 1,000-class head, and train only a small new head for our 38 classes. This usually gives better accuracy with less data and time. **GlobalAveragePooling2D** compresses each feature map into one number so the head stays small.

## 13. Building the Model

In [ ]:
# Section 13a - Model A: a small CNN from scratch
def build_scratch_cnn():
    return keras.Sequential([
        layers.Input(shape=(IMG, IMG, 3)),
        augment,
        layers.Rescaling(1.0 / 255),
        layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"), layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(NUM_CLASSES, activation="softmax"),
    ], name="scratch_cnn")

scratch = build_scratch_cnn()
scratch.summary()

In [ ]:
# Section 13b - Model B: MobileNetV2 transfer learning
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
base.trainable = False                      # freeze the pre-trained layers

transfer = keras.Sequential([
    layers.Input(shape=(IMG, IMG, 3)),
    augment,
    layers.Rescaling(1.0 / 127.5, offset=-1),   # MobileNetV2 expects values in [-1, 1]
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="mobilenetv2_transfer")

transfer.summary()
trainable = sum(int(np.prod(w.shape)) for w in transfer.trainable_weights)
print("Trainable parameters in the transfer model:", f"{trainable:,}", "(only the new head is trained)")

for m in (scratch, transfer):
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

## 14. Training the Model

Watch `val_accuracy` (accuracy on the validation set). `EarlyStopping` stops when `val_loss` stops improving and restores the best weights.

In [ ]:
# Section 14 - train both models
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

print("Training the CNN from scratch ...")
hist_scratch = scratch.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[early], verbose=2)

print("\nTraining the MobileNetV2 transfer model ...")
hist_transfer = transfer.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early], verbose=2)

## 15. Model Evaluation

In [ ]:
# Section 15 - evaluate on the unseen TEST set
results = {}
for name, m in [("CNN from scratch", scratch), ("MobileNetV2 transfer", transfer)]:
    loss, acc = m.evaluate(test_ds, verbose=0)
    results[name] = (loss, acc)
    print(f"{name:22s} test accuracy = {acc:.4f} | test loss = {loss:.4f}")

best_name = max(results, key=lambda k: results[k][1])
best_model = transfer if best_name.startswith("Mobile") else scratch
print("\nBest model:", best_name)

probs_test = best_model.predict(test_ds, verbose=0)
pred_test = probs_test.argmax(axis=1)
present = sorted(set(y_te))                      # only classes that occur in the test set
print(classification_report(y_te, pred_test, labels=present, target_names=[pretty_names[i] for i in present],
                            digits=3, zero_division=0))

**Reading the report:** *precision* = when the model says 'Tomato - Late blight', how often is it right? *Recall* = of the real late-blight leaves, how many did it catch? In agriculture, low recall on a disease class is costly, because sick plants would be called healthy. The numbers come from your own run.

## 16. Data Visualization

In [ ]:
# Section 16a - accuracy and loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h, style in [("Scratch CNN", hist_scratch, "--"), ("MobileNetV2", hist_transfer, "-")]:
    axes[0].plot(h.history["accuracy"], style, label=name + " train")
    axes[0].plot(h.history["val_accuracy"], style, label=name + " validation")
    axes[1].plot(h.history["loss"], style, label=name + " train")
    axes[1].plot(h.history["val_loss"], style, label=name + " validation")
axes[0].set_title("Accuracy"); axes[1].set_title("Loss")
for ax in axes: ax.set_xlabel("epoch"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Section 16b - confusion matrix (38 x 38) and the weakest classes
cm = confusion_matrix(y_te, pred_test, labels=list(range(NUM_CLASSES)))
fig, ax = plt.subplots(figsize=(13, 12))
ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False, include_values=False)
ax.set_title("Confusion matrix of " + best_name + " (rows = true class, columns = predicted)")
plt.show()

per_class = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
worst = np.argsort(per_class)[:8]
print("Classes with the lowest per-class accuracy:")
for i in worst:
    print(f"  {pretty_names[i]:45s} {per_class[i]:.0%}  ({cm[i].sum()} test images)")

A bright diagonal means most predictions are right. Bright cells *off* the diagonal show which classes get confused, usually diseases that look alike on the same crop.

In [ ]:
# Section 16c - prediction examples (green = correct, red = wrong)
fig, axes = plt.subplots(3, 6, figsize=(15, 8.5))
for ax, i in zip(axes.ravel(), rng.choice(len(X_te), 18, replace=False)):
    ok = pred_test[i] == y_te[i]
    ax.imshow(X_te[i]); ax.axis("off")
    ax.set_title(f"true: {pretty_names[y_te[i]]}\npred: {pretty_names[pred_test[i]]} ({probs_test[i].max():.0%})",
                 fontsize=7, color="green" if ok else "red")
plt.tight_layout(); plt.show()

## 17. Testing with New Inputs

Run the next cell and **upload a leaf photo** (JPG or PNG). Best results come from a single leaf on a plain background. If you do not upload anything, a test-set image is used so that the cell still runs.

> Remember: this classifier is **educational** and only knows 38 classes. A photo of anything else will still receive an answer, so check the confidence.

In [ ]:
# Section 17 - classify an uploaded leaf image
from PIL import Image

def predict_leaf(path, model=best_model, top_k=3):
    try:
        img = Image.open(path).convert("RGB")
    except Exception as e:
        print("Could not open this file as an image:", e); return
    arr = np.asarray(img.resize((IMG, IMG)), dtype="float32")[np.newaxis]
    probs = model.predict(arr, verbose=0)[0]
    top = np.argsort(probs)[::-1][:top_k]

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].imshow(img); axes[0].axis("off"); axes[0].set_title("Your image")
    axes[1].barh([pretty_names[i] for i in top][::-1], [probs[i] for i in top][::-1], color="seagreen")
    axes[1].set_xlim(0, 1); axes[1].set_title("Top predictions (confidence)")
    plt.tight_layout(); plt.show()

    crop, _, condition = pretty_names[top[0]].partition(" - ")
    print(f"Predicted plant    : {crop}")
    print(f"Predicted category : {condition or '(not specified)'}")
    print(f"Confidence         : {probs[top[0]]:.1%}")
    if probs[top[0]] < 0.60:
        print("WARNING: low confidence. The image may be unclear, not a leaf, or not one of the 38 classes.")
    print("Educational tool only - not a substitute for expert agricultural diagnosis.")

path = None
try:
    from google.colab import files
    print("Choose a leaf image to upload (or cancel to use a sample).")
    uploaded = files.upload()
    if uploaded:
        path = next(iter(uploaded))
except Exception as e:
    print("No upload available:", type(e).__name__)

if path is None:
    Image.fromarray(X_te[0]).save("sample_leaf.png")
    path = "sample_leaf.png"
    print(f"Using a test-set sample (true label: {pretty_names[y_te[0]]})")
predict_leaf(path)

## 18. Saving the Trained Model

In [ ]:
# Section 18 - save the model and its class names
best_model.save("plant_disease_model.keras")
with open("class_names.json", "w") as f:
    json.dump(pretty_names, f)

reloaded = keras.models.load_model("plant_disease_model.keras")
same = np.allclose(reloaded.predict(X_te[:8].astype("float32"), verbose=0),
                   best_model.predict(X_te[:8].astype("float32"), verbose=0), atol=1e-4)
print("Reloaded model gives the same predictions:", same)
try:
    from google.colab import files
    files.download("plant_disease_model.keras"); files.download("class_names.json")
except ImportError:
    print("(Not running in Colab - the files are in the current folder.)")

## 19. Final Output

In [ ]:
# Section 19 - final summary
print("=" * 62); print("PLANT DISEASE DETECTION - FINAL SUMMARY"); print("=" * 62)
print(f"Classes: {NUM_CLASSES} | images used: {len(X)} ({MAX_PER_CLASS} max per class) | image size: {IMG}x{IMG}")
for name, (loss, acc) in results.items():
    print(f"{name:22s} test accuracy: {acc:.4f}")
print("Best model:", best_name)
print("Reminder: educational classifier trained on lab-style photos; not expert diagnosis.")

## 20. How the Complete Project Works

1. TensorFlow Datasets downloads PlantVillage; we keep up to 200 resized images per class.
2. A stratified split creates train, validation and test sets.
3. Augmentation (flip, rotate, zoom) makes training images more varied.
4. A from-scratch CNN learns features from only our small dataset.
5. MobileNetV2 reuses features learned on ImageNet; only a small new head is trained.
6. Both are scored on the untouched test set; curves, a confusion matrix and a report explain the results.
7. An uploaded leaf is resized, passed through the model and returned as the top-3 classes with confidence, plus a low-confidence warning and a disclaimer.

## 21. Limitations

* **Lab-style images:** PlantVillage photos have plain backgrounds. Accuracy on field photos (soil, hands, shadows, several leaves) is usually much lower.
* Only 38 crop–condition classes. Other crops or diseases are forced into one of them.
* We use a subset of the data and a small image size to keep training quick.
* Confidence is not proof; a model can be confidently wrong.
* **Not a substitute for expert agricultural diagnosis.**

## 22. Real-World Applications

Mobile apps that give farmers an initial screening · crop-monitoring drones · agri-extension training tools · research on early disease detection. Real deployments need field-collected data and expert validation.

## 23. Future Enhancements

* **Fine-tune** the last MobileNetV2 blocks (unfreeze with a very small learning rate).
* Use more images per class and a 224×224 input.
* Collect field photos and test **robustness** to backgrounds and lighting.
* Add **Grad-CAM** heat-maps to show which part of the leaf drove the prediction.
* Convert to **TensorFlow Lite** for on-phone use, or build a Gradio demo.

## 24. Student Mini Challenges

1. Change `MAX_PER_CLASS` to 50 and to 400. How do accuracy and training time change for both models?
2. Remove the `augment` layer from the scratch CNN. Does validation accuracy drop?
3. Unfreeze the last 20 layers of `base` (`for l in base.layers[-20:]: l.trainable = True`), compile with learning rate `1e-5` and train 3 more epochs.
4. Find the two classes that are confused most often in the confusion matrix and look at example images.
5. Photograph a leaf with your phone against a messy background. How does confidence change?
6. Upload a photo of a non-leaf object. What does the model say, and why is that a problem?